# 🦉 Strixlibe × Google Colab 高速GPU LLM Server (進捗ログ可視化版)
自律型ペンテストエージェント「Strixlibe」向けに、Google Colab（**L4 / T4 / G4 GPU**）上で、オープンソースモデル（DeepSeek-R1 14B/32B AWQ 等）を vLLM でホストし、Cloudflare Tunnel 経由で連携するワンクリック・ノートブックです。

### ⚡ 改善点
- **リアルタイムログ表示**: ダウンロード進捗や起動ログが画面に直接表示され、固まる問題を完全解消
- **爆速ロード (14B AWQ 既定)**: 約 8.5GB のため、**実測 1〜2 分** でダウンロード完了
- **推奨GPU**: **L4 GPU** または **T4 GPU**

### 1. 設定項目（フォーム）

In [ ]:
#@title 🚀 サーバー基本設定 { run: "auto" }
MODEL_NAME = "casperhansen/deepseek-r1-distill-qwen-14b-awq" #@param ["casperhansen/deepseek-r1-distill-qwen-14b-awq", "Qwen/Qwen2.5-Coder-14B-Instruct-AWQ", "casperhansen/deepseek-r1-distill-qwen-32b-awq", "Qwen/Qwen2.5-Coder-32B-Instruct-AWQ", "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"]
MAX_MODEL_LEN = 16384 #@param [8192, 16384, 32768] {type:"raw"}
DISPATCHER_URL = "https://strix-dispatcher-2acg5ejyka-uw.a.run.app" #@param {type:"string"}
DISPATCHER_API_KEY = "" #@param {type:"string"}
SLACK_WEBHOOK_URL = "" #@param {type:"string"}
IDLE_TIMEOUT_MINUTES = 20 #@param {type:"integer"}
TARGET_ENV = "all" #@param ["all", "staging", "production"]

### 2. GPU確認とパッケージセットアップ

In [ ]:
# GPUの確認
!nvidia-smi

# 依存パッケージの導入
!pip install -q --no-warn-conflicts vllm hf_transfer || true

# cloudflared のダウンロード
!curl -s -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# vLLM の動作確認
try:
    import vllm
    print(f"\n[+] vLLM {vllm.__version__} is successfully loaded and ready!")
except Exception as e:
    print(f"\n[!] vLLM verification notice: {e}")

### 3. vLLM サーバーと Cloudflare Tunnel の起動（ログ表示付き）

In [ ]:
import subprocess
import time
import re
import requests
import threading
import os

# 過去のプロセスが残っていれば停止
!pkill -f vllm || true
!pkill -f cloudflared || true
time.sleep(2)

log_file = open("vllm_server.log", "w")

print(f"[*] Launching vLLM with model: {MODEL_NAME}...")
print(f"[*] Max Context Length: {MAX_MODEL_LEN}")

vllm_cmd = [
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_NAME,
    "--quantization", "awq",
    "--port", "8000",
    "--host", "0.0.0.0",
    "--trust-remote-code",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", "0.85"
]

# ファイルにログを出力してバッファ詰まりを防止
vllm_proc = subprocess.Popen(vllm_cmd, stdout=log_file, stderr=subprocess.STDOUT)

# ログ監視とヘルスチェック
print("[*] Monitoring vLLM startup log (live tail below):\n")
print("=" * 60)

last_pos = 0
ready = False
start_time = time.time()

while time.time() - start_time < 600:  # 最大10分待機
    # ログファイルの差分を読み出して表示
    if os.path.exists("vllm_server.log"):
        with open("vllm_server.log", "r") as f:
            f.seek(last_pos)
            new_lines = f.read()
            if new_lines:
                print(new_lines, end="")
                last_pos = f.tell()
    
    # プロセスが異常終了していないかチェック
    if vllm_proc.poll() is not None:
        print(f"\n[-] ERROR: vLLM process exited unexpectedly with code {vllm_proc.returncode}.")
        break
        
    # ヘルスチェックエンドポイントの確認
    try:
        r = requests.get("http://localhost:8000/v1/models", timeout=2)
        if r.status_code == 200:
            elapsed = round(time.time() - start_time, 1)
            print("=" * 60)
            print(f"\n[+] vLLM is healthy and ready to serve! (Startup time: {elapsed}s)")
            ready = True
            break
    except:
        pass
        
    time.sleep(3)

if not ready:
    raise RuntimeError("vLLM failed to start within timeout. Check the log output above.")

# Cloudflare Quick Tunnel の起動
print("\n[*] Establishing Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

tunnel_url = None
while True:
    line = tunnel_proc.stdout.readline()
    if not line:
        break
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        print(f"\n🎉 ==========================================================")
        print(f"[+] Cloudflare Tunnel established successfully!")
        print(f"[+] Public URL: {tunnel_url}")
        print(f"[+] OpenAI API Base: {tunnel_url}/v1")
        print(f"==========================================================\n")
        break

api_base = f"{tunnel_url}/v1"

### 4. 疎通テスト（トンネル経由での推論テスト）

In [ ]:
print(f"[*] Sending test prompt to: {api_base}/chat/completions ...")
test_payload = {
    "model": MODEL_NAME,
    "messages": [
        {"role": "user", "content": "Strixペンテストエージェントからの疎通確認テストです。接続が正常であることを、日本語で1〜2文で応答してください。"}
    ],
    "max_tokens": 150,
    "temperature": 0.6
}

try:
    resp = requests.post(f"{api_base}/chat/completions", json=test_payload, timeout=30)
    if resp.status_code == 200:
        result = resp.json()
        reply = result["choices"][0]["message"]["content"]
        print("\n[+] 推論テスト成功！モデルからの返答:")
        print("--------------------------------------------------")
        print(reply.strip())
        print("--------------------------------------------------")
    else:
        print(f"[-] テスト失敗: HTTP {resp.status_code} - {resp.text}")
except Exception as e:
    print(f"[-] テスト例外: {e}")

### 5. Dispatcher 自動同期 & Slack 通知

In [ ]:
# Dispatcher へのセッション登録
if DISPATCHER_URL and DISPATCHER_API_KEY:
    try:
        reg_payload = {
            "provider": "colab-vllm",
            "base_url": api_base,
            "model_name": MODEL_NAME,
            "env_target": TARGET_ENV,
            "ttl_minutes": 120
        }
        headers = {"X-API-Key": DISPATCHER_API_KEY, "Content-Type": "application/json"}
        reg_res = requests.post(f"{DISPATCHER_URL}/api/v1/llm-session", json=reg_payload, headers=headers, timeout=10)
        print(f"[+] Registered to Dispatcher: status {reg_res.status_code}")
    except Exception as e:
        print(f"[-] Dispatcher registration skipped or failed: {e}")
else:
    print("[*] Dispatcher 未設定のため同期をスキップしました（手動接続可能モード）。")

# Slack 通知
if SLACK_WEBHOOK_URL:
    try:
        msg = (
            f"🚀 *Strixlibe Colab GPU LLM Server is ONLINE!*\n"
            f"• *Model*: `{MODEL_NAME}`\n"
            f"• *Endpoint*: `{api_base}`\n"
            f"• *Env*: `{TARGET_ENV}`\n"
            f"• *Watchdog*: {IDLE_TIMEOUT_MINUTES} mins auto-kill\n"
            f"💡 Strixlibe will automatically route scans to this model."
        )
        requests.post(SLACK_WEBHOOK_URL, json={"text": msg}, timeout=5)
        print("[+] Slack notification sent.")
    except Exception as e:
        print(f"[-] Slack notification failed: {e}")

### 6. CU保護ウォッチドッグ（バックグラウンド監視・自動シャットダウン）

In [ ]:
from google.colab import runtime

def watchdog_worker():
    print(f"[*] Watchdog background worker started. Auto-terminate after {IDLE_TIMEOUT_MINUTES}m inactivity.")
    last_seen_time = time.time()
    idle_timeout_seconds = IDLE_TIMEOUT_MINUTES * 60
    
    while True:
        time.sleep(30)
        elapsed = time.time() - last_seen_time
        if elapsed > idle_timeout_seconds:
            print(f"[!] Inactivity threshold ({IDLE_TIMEOUT_MINUTES} min) exceeded. Terminating runtime to preserve Compute Units.")
            if SLACK_WEBHOOK_URL:
                try:
                    requests.post(SLACK_WEBHOOK_URL, json={"text": "🛑 *Colab LLM Server disconnected* (Idle timeout). Auto-fallback to OpenCode(Zen) engaged."}, timeout=5)
                except:
                    pass
            runtime.unassign()
            break

# バックグラウンドスレッドとして起動
t = threading.Thread(target=watchdog_worker, daemon=True)
t.start()
print("[OK] All systems operational! Tunnel is active and watchdog is monitoring.")